# Partial and Full Fine-Tuning — ResNet50 / HAM10000

Run the dataset download and metadata cells (1–8) in the original notebook first. This notebook continues from `df` and uses the same image folders. We split by `lesion_id` to reduce leakage from multiple images of the same lesion. Note: the original binary labels are a teaching simplification, not a clinically valid malignancy definition.


In [ ]:
!pip install -q kaggle

In [ ]:
from google.colab import files

files.upload()

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/kaggle.json
!chmod 600 ~/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d kmader/skin-cancer-mnist-ham10000

In [ ]:
!unzip -q skin-cancer-mnist-ham10000.zip -d /content/skin_data

In [ ]:
import pandas as pd

df = pd.read_csv(
    "/content/skin_data/HAM10000_metadata.csv"
)

print(df.head())
print(df["dx"].value_counts())

In [ ]:
print(df.columns)

In [ ]:
df = df[df["dx"] != "akiec"].copy()

df["label"] = df["dx"].apply(
    lambda x: 1 if x in ["mel", "bcc"] else 0
)

print(df["label"].value_counts())

## Prepare image datasets and lesion-level train/test split

In [ ]:
import numpy as np
import tensorflow as tf
from pathlib import Path
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report
from keras.applications.resnet50 import ResNet50, preprocess_input
from keras.utils import load_img, img_to_array
from keras import layers, Model

tf.keras.utils.set_random_seed(42)
image_dirs = [Path('/content/skin_data/ham10000_images_part_1'), Path('/content/skin_data/ham10000_images_part_2')]
image_paths = {p.stem: str(p) for folder in image_dirs for p in folder.glob('*.jpg')}
work_df = df.copy()  # df from your existing metadata + label cells
work_df['path'] = work_df['image_id'].map(image_paths)
work_df = work_df.dropna(subset=['path']).reset_index(drop=True)
assert 'lesion_id' in work_df.columns, 'HAM10000 metadata must contain lesion_id'

# Split by lesion, not by image: related images must not be in both splits.
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(work_df, work_df['label'], groups=work_df['lesion_id']))
train_df = work_df.iloc[train_idx].reset_index(drop=True)
test_df = work_df.iloc[test_idx].reset_index(drop=True)
print('Train:', len(train_df), 'Test:', len(test_df))
print('Train labels:', train_df['label'].value_counts().to_dict())
print('Test labels:', test_df['label'].value_counts().to_dict())

IMG_SIZE = (224, 224)
BATCH_SIZE = 16
def load_and_preprocess(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, IMG_SIZE)
    img = tf.cast(img, tf.float32)
    img = preprocess_input(img)
    return img, tf.cast(label, tf.float32)

def make_dataset(frame, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((frame['path'].values, frame['label'].values.astype('float32')))
    if shuffle:
        ds = ds.shuffle(len(frame), seed=42)
    return ds.map(load_and_preprocess, num_parallel_calls=tf.data.AUTOTUNE).batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

train_ds = make_dataset(train_df, shuffle=True)
test_ds = make_dataset(test_df)


## Common starting point: train the classifier head

In [ ]:
# Step 1: train only the new classifier head (same starting point for both experiments)
backbone = ResNet50(weights='imagenet', include_top=False, pooling='avg', input_shape=(224, 224, 3))
backbone.trainable = False
inputs = layers.Input(shape=(224, 224, 3))
features = backbone(inputs, training=False)  # keep BatchNorm running stats stable
outputs = layers.Dense(1, activation='sigmoid')(features)
model = Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss='binary_crossentropy', metrics=['accuracy'])
model.fit(train_ds, epochs=2, validation_data=test_ds)
model.save_weights('/content/head_start.weights.h5')


## Partial fine-tuning

In [ ]:
# PARTIAL FINE-TUNING: train the last 20 backbone layers + classifier
backbone.trainable = True
for layer in backbone.layers[:-20]:
    layer.trainable = False
for layer in backbone.layers[-20:]:
    layer.trainable = not isinstance(layer, layers.BatchNormalization)

# Always recompile AFTER changing trainable flags.
model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='binary_crossentropy', metrics=['accuracy'])
print('Trainable backbone layers:', sum(int(layer.trainable) for layer in backbone.layers))
model.fit(train_ds, epochs=3, validation_data=test_ds)
partial_predictions = (model.predict(test_ds).ravel() >= 0.5).astype(int)
print(classification_report(test_df['label'], partial_predictions, labels=[0,1], target_names=['Benign','Malignant'], zero_division=0))
model.save('/content/partial_finetuned.keras')


## Full fine-tuning

In [ ]:
# FULL FINE-TUNING: reset to the same head-only checkpoint for a fair comparison
# Rebuild the architecture with fresh ImageNet weights, then load the trained head.
full_backbone = ResNet50(weights='imagenet', include_top=False, pooling='avg', input_shape=(224, 224, 3))
full_backbone.trainable = False
full_inputs = layers.Input(shape=(224, 224, 3))
full_features = full_backbone(full_inputs, training=False)
full_outputs = layers.Dense(1, activation='sigmoid')(full_features)
full_model = Model(full_inputs, full_outputs)
full_model.load_weights('/content/head_start.weights.h5')

full_backbone.trainable = True
# BatchNorm layers stay frozen for stability with small batches.
for layer in full_backbone.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

full_model.compile(optimizer=tf.keras.optimizers.Adam(1e-5), loss='binary_crossentropy', metrics=['accuracy'])
print('Trainable backbone layers:', sum(int(layer.trainable) for layer in full_backbone.layers))
full_model.fit(train_ds, epochs=3, validation_data=test_ds)
full_predictions = (full_model.predict(test_ds).ravel() >= 0.5).astype(int)
print(classification_report(test_df['label'], full_predictions, labels=[0,1], target_names=['Benign','Malignant'], zero_division=0))
full_model.save('/content/full_finetuned.keras')


## Test a new image

In [ ]:
# Predict a single image using either fine-tuned model (no SVM)
from google.colab import files
from keras.models import load_model
uploaded = files.upload()
filename = next(iter(uploaded))
chosen_model = load_model('/content/partial_finetuned.keras')  # or full_finetuned.keras
img = img_to_array(load_img(filename, target_size=IMG_SIZE))
img = preprocess_input(np.expand_dims(img, axis=0))
score = float(chosen_model.predict(img, verbose=0)[0][0])
print('Predicted:', 'Malignant' if score >= 0.5 else 'Benign', '| score:', round(score, 4))
